# Treinamento e comparação de modelos sobre a ABT

Este notebook compara três classificadores para prever `target` a partir de `Dados/abt.csv`: regressão logística, árvore de decisão e random forest. O pré-processamento é aprendido somente com os dados de treino e fica incorporado ao artefato final.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

required_modules = ["matplotlib", "numpy", "pandas", "sklearn"]
missing_modules = [module for module in required_modules if importlib.util.find_spec(module) is None]

print(f"Python/kernel em uso: {sys.executable}")

if missing_modules:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    requirements_path = next(
        (path / "requirements.txt" for path in candidates if (path / "requirements.txt").is_file()),
        None,
    )
    if requirements_path is None:
        raise FileNotFoundError("requirements.txt não encontrado. Execute o notebook a partir de data-platform ou da raiz do projeto.")

    print(f"Instalando dependências ausentes no kernel atual: {missing_modules}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
else:
    print("Dependências principais já disponíveis no kernel atual.")

In [ ]:
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
DECISION_THRESHOLD = 0.50
CV_FOLDS = 5

## 1. Carregamento da ABT

A busca funciona ao executar o notebook pela pasta `Model`, por `data-platform` ou pela raiz do repositório.

In [ ]:
project_root = Path.home() / "Desktop/ProjetoFinalIA_Andre/ProjetoFinalPosFIA"
csv_candidates = [
    project_root / "data-platform/Dados/abt.csv",
    Path("../Dados/abt.csv"),
    Path("Dados/abt.csv"),
    Path("data-platform/Dados/abt.csv"),
]

ABT_PATH = None
searched = []
for path in csv_candidates:
    try:
        candidate = path if path.is_absolute() else path.absolute()
        searched.append(str(candidate))
        if candidate.is_file():
            ABT_PATH = candidate
            break
    except OSError as error:
        searched.append(f"{path} (indisponível: {error})")

if ABT_PATH is None:
    raise FileNotFoundError("abt.csv não encontrado. Caminhos pesquisados:\n" + "\n".join(searched))

df = pd.read_csv(ABT_PATH, encoding="utf-8")
print(f"Arquivo: {ABT_PATH}")
print(f"ABT carregada: {df.shape[0]:,} linhas e {df.shape[1]} colunas")
display(df.head())

## 2. Preparação das variáveis

O identificador `sk_id_curr` não participa do treinamento. Valores infinitos são tratados como ausentes e imputados dentro do pipeline.

In [ ]:
required_columns = {"sk_id_curr", "target"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Colunas obrigatórias ausentes na ABT: {sorted(missing_columns)}")

df_model = df.replace([np.inf, -np.inf], np.nan).copy()
X = df_model.drop(columns=["target", "sk_id_curr"])
y = pd.to_numeric(df_model["target"], errors="raise").astype(int)

empty_columns = X.columns[X.isna().all()].tolist()
X = X.drop(columns=empty_columns)

print(f"Variáveis explicativas: {X.shape[1]}")
print(f"Colunas totalmente vazias removidas: {empty_columns}")
print("Distribuição do target (%):")
display(y.value_counts(normalize=True).mul(100).round(2).sort_index())

## 3. Separação entre treino e teste

A estratificação mantém aproximadamente a mesma proporção de inadimplentes nos dois conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

print(f"Treino: {len(X_train):,} | Teste: {len(X_test):,}")
print(f"Numéricas: {len(numeric_features)} | Categóricas: {len(categorical_features)}")

## 4. Pré-processamento e comparação de modelos

Medianas, escalas e categorias são aprendidas somente dentro de cada partição de treino da validação cruzada. A comparação usa o mesmo split, o mesmo pré-processamento e as mesmas métricas para os três modelos. A métrica principal de seleção é ROC AUC, pois a base é desbalanceada e o objetivo é ordenar clientes por risco.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

candidate_models = {
    "logistic_regression": LogisticRegression(
        class_weight="balanced",
        max_iter=100,
        solver="newton-cholesky",
        random_state=RANDOM_STATE,
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=100,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=100,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

results = []
fitted_models = {}

for model_name, classifier in candidate_models.items():
    estimator = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])
    started_at = time.perf_counter()
    cv_scores = cross_val_score(estimator, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=-1)
    estimator.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - started_at

    y_probability_model = estimator.predict_proba(X_test)[:, 1]
    y_prediction_model = (y_probability_model >= DECISION_THRESHOLD).astype(int)

    results.append({
        "model": model_name,
        "roc_auc": roc_auc_score(y_test, y_probability_model),
        "accuracy": accuracy_score(y_test, y_prediction_model),
        "precision": precision_score(y_test, y_prediction_model, zero_division=0),
        "average_precision": average_precision_score(y_test, y_probability_model),
        "cv_roc_auc_mean": cv_scores.mean(),
        "cv_roc_auc_std": cv_scores.std(),
        "fit_seconds": round(fit_seconds, 2),
    })
    fitted_models[model_name] = estimator

comparison = pd.DataFrame(results).sort_values(["roc_auc", "precision", "accuracy"], ascending=False)
best_model_name = comparison.iloc[0]["model"]
model = fitted_models[best_model_name]

print(f"Melhor modelo pela ROC AUC: {best_model_name}")
display(comparison.round(4))

## 5. Avaliação do melhor modelo

ROC AUC mede a capacidade geral de ordenação. Accuracy e precision entram na comparação, mas ROC AUC é priorizada porque a base é desbalanceada.

In [ ]:
y_probability = model.predict_proba(X_test)[:, 1]
y_prediction = (y_probability >= DECISION_THRESHOLD).astype(int)

roc_auc = roc_auc_score(y_test, y_probability)
accuracy = accuracy_score(y_test, y_prediction)
precision_model = precision_score(y_test, y_prediction, zero_division=0)
average_precision = average_precision_score(y_test, y_probability)

print(f"Modelo selecionado: {best_model_name}")
print(f"ROC AUC: {roc_auc:.4f} | Accuracy: {accuracy:.4f} | Precision: {precision_model:.4f} | Average Precision: {average_precision:.4f}")
print(f"\nRelatório de classificação (limiar = {DECISION_THRESHOLD:.2f}):")
print(classification_report(y_test, y_prediction, digits=4))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_prediction,
    display_labels=["Adimplente", "Inadimplente"],
    cmap="Blues",
)
plt.title("Matriz de confusão")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_probability)
precision, recall, _ = precision_recall_curve(y_test, y_probability)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="Curva ROC", xlabel="Taxa de falsos positivos", ylabel="Taxa de verdadeiros positivos")
axes[0].legend()

axes[1].plot(recall, precision, label=f"AP = {average_precision:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", label="Modelo aleatório")
axes[1].set(title="Curva Precision-Recall", xlabel="Recall", ylabel="Precisão")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Variáveis mais influentes

Quando o melhor modelo é linear, os coeficientes indicam associação com a pontuação de inadimplência. Quando é baseado em árvore, usamos a importância das variáveis do próprio modelo. Em ambos os casos, interpretabilidade não significa causalidade.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
classifier = model.named_steps["classifier"]

if hasattr(classifier, "coef_"):
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": classifier.coef_[0],
        "absolute_importance": np.abs(classifier.coef_[0]),
    }).sort_values("absolute_importance", ascending=False)
elif hasattr(classifier, "feature_importances_"):
    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": classifier.feature_importances_,
        "absolute_importance": classifier.feature_importances_,
    }).sort_values("absolute_importance", ascending=False)
else:
    importance = pd.DataFrame(columns=["feature", "importance", "absolute_importance"])

display(importance.head(25))

## 7. Persistência do modelo

O artefato contém pré-processamento, classificador, limiar e metadados necessários para inferência.

In [ ]:
ARTIFACT_DIR = ABT_PATH.parent.parent / "Model/artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "logistic_regression_abt.pkl"
COMPARISON_PATH = ARTIFACT_DIR / "model_comparison.csv"

artifact = {
    "model": model,
    "model_name": best_model_name,
    "decision_threshold": DECISION_THRESHOLD,
    "input_features": X.columns.tolist(),
    "metrics": {
        "roc_auc": roc_auc,
        "accuracy": accuracy,
        "precision": precision_model,
        "average_precision": average_precision,
        "cv_roc_auc": comparison.iloc[0]["cv_roc_auc_mean"],
    },
    "model_comparison": comparison.to_dict(orient="records"),
    "cv_folds": CV_FOLDS,
}
with MODEL_PATH.open("wb") as file:
    pickle.dump(artifact, file)
comparison.to_csv(COMPARISON_PATH, index=False)
print(f"Modelo salvo em: {MODEL_PATH.resolve()}")